# EDA

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.duplicated().sum()

0

In [5]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

# Transform

In [7]:
# ubah nama kolom dataset menyesuaikan data modelling
df.rename(columns = {
    'customerID' : 'customer_id',
    'SeniorCitizen': 'senior_citizen',
    'Partner' : 'partner',
    'Dependents' : 'dependents',
    'PaperlessBilling': 'paperless_billing',
    'PaymentMethod': 'payment_method',
    'MonthlyCharges': 'monthly_charges',
    'TotalCharges': 'total_charges',
    'Churn' : 'churn_status',
    'PhoneService': 'phone_service',
    'MultipleLines': 'multiple_lines',
    'InternetService': 'internet_service',
    'OnlineSecurity': 'online_security',
    'OnlineBackup': 'online_backup',
    'DeviceProtection': 'device_protection',
    'TechSupport': 'tech_support',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_movies',
    'Contract' : 'contract'
}, inplace=True)

In [8]:
# ubah string kosong menjadi 0 pada kolom total_charges
df['total_charges'] = df['total_charges'].replace(' ', 0)
df['total_charges'] = df['total_charges'].fillna(0)

In [9]:
# ubah value yes dan no menjadi 1 dan 0 pada kolom churn status
df['churn_status'] = df['churn_status'].apply(lambda x : 1 if x == 'Yes' else 0)

In [10]:
# ganti tipe data kolom
df['total_charges'] = df['total_charges'].astype(float)

## Dim tables

### Dim Customer

In [11]:
dim_customer = df[['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents']]

### Dim Service

In [12]:
dim_service = df[['phone_service', 'multiple_lines', 'internet_service', 'online_security', 
                  'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies'
                ]].drop_duplicates().reset_index(drop = True)

dim_service['service_id'] = dim_service.index + 1

### Dim Billing

In [13]:
dim_billing = df[['paperless_billing', 'payment_method']].drop_duplicates().reset_index(drop = True)
dim_billing['billing_id'] = dim_billing.index + 1

### Dim Contract

In [14]:
dim_contract = df[['contract']].drop_duplicates().reset_index(drop = True)
dim_contract['contract_id'] = dim_contract.index + 1

## Fact Tables

In [15]:
# Join data asli dengan tabel dimensi untuk mendapatkan ID-nya
fact_subscription = df.merge(dim_billing, on=['paperless_billing', 'payment_method']) \
                      .merge(dim_contract, left_on='contract', right_on='contract') \
                      .merge(dim_service, on=['phone_service', 'multiple_lines', ''
                                'internet_service', 'online_security', 
                                    'online_backup', 'device_protection', 'tech_support', 
                                    'streaming_tv', 'streaming_movies'
                            ])


fact_subscription = fact_subscription[[
    'customer_id', 'contract_id', 'service_id', 'billing_id',
    'tenure', 'monthly_charges', 'total_charges', 'churn_status'
]]

fact_subscription.insert(0, 'subscription_id', range(1, len(fact_subscription) + 1))

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   object 
 1   gender             7043 non-null   object 
 2   senior_citizen     7043 non-null   int64  
 3   partner            7043 non-null   object 
 4   dependents         7043 non-null   object 
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   object 
 7   multiple_lines     7043 non-null   object 
 8   internet_service   7043 non-null   object 
 9   online_security    7043 non-null   object 
 10  online_backup      7043 non-null   object 
 11  device_protection  7043 non-null   object 
 12  tech_support       7043 non-null   object 
 13  streaming_tv       7043 non-null   object 
 14  streaming_movies   7043 non-null   object 
 15  contract           7043 non-null   object 
 16  paperless_billing  7043 

# GX

## Data Validation

In [17]:
from great_expectations.data_context import FileDataContext # import filedatacontext

context = FileDataContext.create(project_root_dir='./') # buat instance baru di folder program

In [18]:
# inisiasi nama datasource
datasource_name = 'telco_churn'

# inisiasi variabel datasource yang menampung namasource dari dataframe pandas
datasource = context.sources.add_pandas(datasource_name)

# inisiasi nama aset
asset_name = 'workflow_churn'

# add dataframe aset ke datasource
asset = datasource.add_dataframe_asset(name=asset_name) 

# inisiasi batch request yang akan dipakai untuk validasi data dari dataframe df
batch_request = asset.build_batch_request(
    dataframe=df
)

# inisiasi nama expectation suite
expectation_suite_name = 'expectation_df'

# buat atau update expectation suite yang ada di dalam context
context.add_or_update_expectation_suite(expectation_suite_name)


# inisiasi validator yang akan memvalidasi data
validator = context.get_validator(
    batch_request = batch_request,
    expectation_suite_name = expectation_suite_name
)

# tampilkan data
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_status
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Expectations

### 1. table to have 21 columns

In [20]:
# cek apakah jumlah kolom pada dataset adalah 18
validator.expect_table_column_count_to_equal(value=21)

Calculating Metrics:   0%|          | 0/3 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": 21
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### 2. to be unique

In [ ]:
# cek apakah di kolom customer_id valuenya bernilai unique
validator.expect_column_values_to_be_unique('customer_id')

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 7043,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### 3. not match regex

In [ ]:
# cek apakah di kolom total_charges tidak ada huruf atau simbol tertentu
validator.expect_column_values_to_not_match_regex(
    column="total_charges",
    regex=r"[[a-zA-Z\-_\@\#\$\%\&\*\(\)\+\=\!\?]"
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 7043,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### 4. to be in type list

In [23]:
# cek apakah tipe data pada kolom monthly_charges ada di antara integer atau float
validator.expect_column_values_to_be_in_type_list('monthly_charges', ['integer', 'float'])

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "float64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### 5. to be in set

In [24]:
# cek apakah pada kolom gender, isi dari valuenya hanya di antara Female dan Male
validator.expect_column_values_to_be_in_set('gender', ['Female', 'Male'])

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 7043,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### save into suite

In [25]:
validator.save_expectation_suite(discard_failed_expectations=False)

### Checkpoint

In [26]:
# inisiasi cekpoin yang dapat melakukan validasi secara terstruktur
checkpoint_1 = context.add_or_update_checkpoint(
    name = 'checkpoint_1',
    validator = validator,
)

In [27]:
# inisasi variabel checkpoin result yang akan menjalankan cekpoin sebelumnya
# berisi informasi apakah validasi yang dilakukan true atau fail
checkpoint_result = checkpoint_1.run()

Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

In [28]:
# buat data docs berisi hasil validasi data menggunakan great expectations
context.build_data_docs()

{'local_site': 'file://e:\\Hacktiv8\\Phase 2\\Final Project\\gx\\uncommitted/data_docs/local_site/index.html'}